In [19]:
# the updated plotting function so that for Pearson and Spearman plots only:
# No y-axis label
# No y-tick labels
# No legend

import os
import re
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = "."  # folder containing txt files
METRICS = ["val_loss", "pearson", "spearman"]

INDEX_TO_NAME = {
    # "10": "HT12",
    # "17": "SpCas9-HF1 NGG",
    # "18": "SpCas9-NG NGG",
}

# ---- filename parser ----
def parse_filename(base):
    name = base.lower()
    if not (name.endswith(".txt")):
        return None
    m_idx = re.search(r'(\d+)', name)
    if not m_idx:
        return None
    idx = m_idx.group(1)
    model = None
    if "cnn1_only" in name:
        model = "CNN1_only"
    elif "full_model" in name:
        model = "full_model"
    else:
        return None
    metric = None
    if "pearson" in name:
        metric = "pearson"
    elif "spearman" in name:
        metric = "spearman"
    elif re.search(r'val[_\-]?loss', name):
        metric = "val_loss"
    if metric is None:
        return None
    return idx, model, metric

def read_numeric_lines(path):
    vals = []
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            try:
                vals.append(float(s))
            except:
                pass
    return vals

rows = []
skipped = []
for path in glob.glob(os.path.join(DATA_DIR, "*")):
    base = os.path.basename(path)
    info = parse_filename(base)
    if not info:
        skipped.append(base)
        continue
    idx, model, metric = info
    values = read_numeric_lines(path)
    if not values:
        skipped.append(base)
        continue
    for v in values:
        rows.append({
            "index": idx,
            "dataset": INDEX_TO_NAME.get(idx, idx),
            "model": model,
            "metric": metric,
            "value": v
        })

if not rows:
    raise RuntimeError(f"No matching files found. Skipped (first 10): {skipped[:10]}")

df = pd.DataFrame(rows)

# Min–max normalization within each (index, metric)
def normalize_group(g):
    vmin = g["value"].min()
    vmax = g["value"].max()
    if math.isclose(vmin, vmax):
        g["norm"] = 0.5
    else:
        g["norm"] = (g["value"] - vmin) / (vmax - vmin)
    return g

df = df.groupby(["index", "metric"], group_keys=False).apply(normalize_group)

# Sort datasets numerically by index, then assign rank starting from 1
sorted_datasets = (
    df[["index", "dataset"]]
    .drop_duplicates()
    .assign(index_num=lambda x: x["index"].astype(int))
    .sort_values("index_num")
)
sorted_datasets["rank"] = range(1, len(sorted_datasets) + 1)
dataset_rank_map = dict(zip(sorted_datasets["dataset"], sorted_datasets["rank"]))

df["rank"] = df["dataset"].map(dataset_rank_map)

# Get dataset order based on natural numeric sorting of index
dataset_order = (
    df[["index", "dataset"]]
    .drop_duplicates()
    .assign(index_num=lambda x: x["index"].astype(int))
    .sort_values("index_num")["dataset"]
    .tolist()
)

# Plot function
def plot_metric(metric_name):
    dsub = df[df["metric"] == metric_name].copy()
    if dsub.empty:
        print(f"[WARN] No data for metric '{metric_name}'. Skipping.")
        return

    display_name = metric_name
    if metric_name.lower() == "pearson":
        display_name = "Pearson"
    elif metric_name.lower() == "spearman":
        display_name = "Spearman"

    plt.figure(figsize=(10, max(6, len(dataset_order) * 0.5)), dpi=600)
    ax = sns.violinplot(
        data=dsub,
        y="dataset", x="norm",
        hue="model",
        hue_order=["CNN1_only", "full_model"], 
        orient="h",
        cut=0,
        scale="width",
        inner="quartile",
        order=dataset_order
    )

    plt.xlabel(
        "Normalized value",
        fontsize=36, labelpad=10
    )
    plt.title(f"{display_name}: CNN1-only vs full model", fontsize=36, pad=15)
    plt.xticks(fontsize=28)

    if metric_name.lower() in ["pearson", "spearman"]:
        ax.set_ylabel(None)              
        ax.set_yticklabels([])           
        ax.tick_params(axis='y', length=0)  
        ax.get_legend().remove()        
    else:
        plt.ylabel("Dataset (Index)", fontsize=36, labelpad=10)
        plt.yticks(fontsize=28)
        plt.legend(
            title="Model", loc="best",
            fontsize=28, title_fontsize=28
        )

    plt.tight_layout()
    plt.show()

# Show all three metrics
for m in METRICS:
    plot_metric(m)




/var/folders/8p/2p_212ld2qvgjsycxk_lmmg00000gn/T/ipykernel_2498/1214938348.py:101: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(["index", "metric"], group_keys=False).apply(normalize_group)
/var/folders/8p/2p_212ld2qvgjsycxk_lmmg00000gn/T/ipykernel_2498/1214938348.py:139: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  ax = sns.violinplot(


/var/folders/8p/2p_212ld2qvgjsycxk_lmmg00000gn/T/ipykernel_2498/1214938348.py:139: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  ax = sns.violinplot(


/var/folders/8p/2p_212ld2qvgjsycxk_lmmg00000gn/T/ipykernel_2498/1214938348.py:139: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  ax = sns.violinplot(
